In [1]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from openpyxl import load_workbook

# ── PATHS ──────────────────────────────────────────────────────────────────
DATA_ROOT  = Path('/Users/harshit/Desktop/KelloggXParlamint/Kellogg/data/results/meta_transcript')
OUTPUT_DIR = DATA_ROOT / 'Analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── DIMENSION → GROUP MAPPING ──────────────────────────────────────────────
DIMENSIONS = {
    'alignment':  ['Coalition', 'Opposition'],
    'gender':     ['Male', 'Female'],
    'generation': ['Silent_Generation', 'Baby_Boomers', 'Generation_X', 'Millennials', 'Generation_Z'],
    'ideology':   ['Left', 'Centre', 'Right'],
}

# All groups flattened (for Excel sheet names)
ALL_GROUPS = [g for groups in DIMENSIONS.values() for g in groups]

print('Data root :', DATA_ROOT)
print('Output dir:', OUTPUT_DIR)
print('Groups    :', ALL_GROUPS)


Data root : /Users/harshit/Desktop/KelloggXParlamint/Kellogg/data/results/meta_transcript
Output dir: /Users/harshit/Desktop/KelloggXParlamint/Kellogg/data/results/meta_transcript/Analysis
Groups    : ['Coalition', 'Opposition', 'Male', 'Female', 'Silent_Generation', 'Baby_Boomers', 'Generation_X', 'Millennials', 'Generation_Z', 'Left', 'Centre', 'Right']


In [2]:
# ── STEP 1: Walk folder tree and collect mean values ──────────────────────
# Path pattern: DATA_ROOT / {country} / {dimension} / {group} / {year}_nature_vs_importance.csv

records = []
skipped = []

country_dirs = sorted([d for d in DATA_ROOT.iterdir() if d.is_dir() and d.name != 'Average'])
print(f'Countries found: {[d.name for d in country_dirs]}')
print(f'Total countries: {len(country_dirs)}')

for country_dir in country_dirs:
    country = country_dir.name
    for dim, groups in DIMENSIONS.items():
        dim_dir = country_dir / dim
        if not dim_dir.exists():
            skipped.append(f'{country}/{dim} — folder not found')
            continue
        for group in groups:
            group_dir = dim_dir / group
            if not group_dir.exists():
                skipped.append(f'{country}/{dim}/{group} — folder not found')
                continue
            csv_files = sorted(group_dir.glob('*_nature_vs_importance.csv'))
            if not csv_files:
                skipped.append(f'{country}/{dim}/{group} — no CSV files')
                continue
            for csv_path in csv_files:
                m = re.match(r'(\d{4})_', csv_path.name)
                if not m:
                    continue
                year = int(m.group(1))
                try:
                    df = pd.read_csv(csv_path, index_col=0)
                    nums = df.select_dtypes(include=[np.number]).to_numpy().flatten()
                    mean_val = float(np.nanmean(nums)) if len(nums) > 0 else np.nan
                    records.append({
                        'Country':   country,
                        'Dimension': dim,
                        'Group':     group,
                        'Year':      year,
                        'MeanValue': mean_val
                    })
                except Exception as e:
                    skipped.append(f'{country}/{dim}/{group}/{csv_path.name} — read error: {e}')

df_all = pd.DataFrame(records)
print(f'\n── CHECKPOINT 1: Data Collection ──')
print(f'Total records collected : {len(df_all):,}')
print(f'Countries               : {df_all["Country"].nunique()}')
print(f'Groups                  : {df_all["Group"].nunique()} → {sorted(df_all["Group"].unique())}')
print(f'Year range              : {df_all["Year"].min()} – {df_all["Year"].max()}')
print(f'Skipped paths           : {len(skipped)}')
if skipped:
    print('  First 10 skipped:')
    for s in skipped[:10]: print(f'    • {s}')


Countries found: ['AT', 'Analysis', 'BA', 'BE', 'BG', 'CZ', 'DK', 'EE', 'ES', 'ES-CT', 'ES-GA', 'ES-PV', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IS', 'IT', 'LV', 'NL', 'NO', 'PL', 'PT', 'RS', 'SE', 'SI', 'TR', 'UA']
Total countries: 30

── CHECKPOINT 1: Data Collection ──
Total records collected : 3,582
Countries               : 29
Groups                  : 12 → ['Baby_Boomers', 'Centre', 'Coalition', 'Female', 'Generation_X', 'Generation_Z', 'Left', 'Male', 'Millennials', 'Opposition', 'Right', 'Silent_Generation']
Year range              : 1996 – 2024
Skipped paths           : 38
  First 10 skipped:
    • AT/generation/Generation_Z — folder not found
    • Analysis/alignment — folder not found
    • Analysis/gender — folder not found
    • Analysis/generation — folder not found
    • Analysis/ideology — folder not found
    • BA/generation/Generation_Z — folder not found
    • BE/generation/Generation_Z — folder not found
    • CZ/generation/Generation_Z — folder not found
    • CZ/ideo

In [3]:
# ── CHECKPOINT 2: Verify coverage per group ────────────────────────────────
print('── CHECKPOINT 2: Records per Group ──')
coverage = df_all.groupby(['Dimension','Group']).agg(
    records=('MeanValue','count'),
    countries=('Country','nunique'),
    years=('Year','nunique'),
    mean_score=('MeanValue', lambda x: round(x.mean(), 4)),
    nulls=('MeanValue', lambda x: x.isna().sum())
).reset_index()
print(coverage.to_string(index=False))

# ── CHECKPOINT 3: Score range report (no assertion) ───────────────────────
print('\n── CHECKPOINT 3: Score range ──')
sample = df_all.groupby('Group')['MeanValue'].describe()[['min','mean','max']].round(4)
print(sample)
print(f'\nOverall range: [{df_all["MeanValue"].min():.6f}, {df_all["MeanValue"].max():.6f}]')
print(f'NaN count    : {df_all["MeanValue"].isna().sum()}')
print('Checkpoint 3 complete ✓')


── CHECKPOINT 2: Records per Group ──
 Dimension             Group  records  countries  years  mean_score  nulls
 alignment         Coalition      357         29     29      0.3142      3
 alignment        Opposition      307         25     29      0.3149      3
    gender            Female      362         29     29      0.3157      2
    gender              Male      362         29     29      0.3184      2
generation      Baby_Boomers      328         25     29      0.3159      3
generation      Generation_X      327         25     29      0.3166      3
generation      Generation_Z       12          6      7      0.3089      5
generation       Millennials      269         25     27      0.3115      9
generation Silent_Generation      255         25     28      0.3008     19
  ideology            Centre      215         22     28      0.3085     11
  ideology              Left      361         29     29      0.3165      3
  ideology             Right      363         29     29      0

In [4]:
# ── STEP 2: Build pivot (Year x Country) per group and save to Excel ──────
# Each pivot is reindexed to the FULL year range and FULL country list so
# missing cells appear as NaN (white) rather than being dropped entirely.

excel_path = OUTPUT_DIR / 'meta_averages_by_group.xlsx'

# Compute global year range and country list across ALL data
all_years     = sorted(df_all['Year'].unique())
all_countries = sorted(df_all['Country'].unique())
print(f'Global year range : {all_years[0]} – {all_years[-1]} ({len(all_years)} years)')
print(f'Global countries  : {len(all_countries)} → {all_countries}')

pivots = {}  # store for plotting later

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    for dim, groups in DIMENSIONS.items():
        for group in groups:
            subset = df_all[(df_all['Dimension'] == dim) & (df_all['Group'] == group)]
            if subset.empty:
                print(f'  [SKIP] {group} — no data')
                continue
            pivot = (
                subset
                .pivot(index='Year', columns='Country', values='MeanValue')
                # Reindex to FULL global range — missing combos become NaN
                .reindex(index=all_years, columns=all_countries)
            )
            pivots[group] = pivot
            sheet_name = group[:31]
            pivot.to_excel(writer, sheet_name=sheet_name)
            filled = pivot.notna().sum().sum()
            total  = pivot.size
            print(f'  Saved sheet: {sheet_name:25s} | {pivot.shape[0]} years x {pivot.shape[1]} countries | filled: {filled}/{total} ({filled/total*100:.1f}%)')

print(f'\n── CHECKPOINT 4: Excel saved ──')
print(f'Path   : {excel_path}')
print(f'Sheets : {list(pivots.keys())}')


Global year range : 1996 – 2024 (29 years)
Global countries  : 29 → ['AT', 'BA', 'BE', 'BG', 'CZ', 'DK', 'EE', 'ES', 'ES-CT', 'ES-GA', 'ES-PV', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IS', 'IT', 'LV', 'NL', 'NO', 'PL', 'PT', 'RS', 'SE', 'SI', 'TR', 'UA']
  Saved sheet: Coalition                 | 29 years x 29 countries | filled: 357/841 (42.4%)
  Saved sheet: Opposition                | 29 years x 29 countries | filled: 307/841 (36.5%)
  Saved sheet: Male                      | 29 years x 29 countries | filled: 362/841 (43.0%)
  Saved sheet: Female                    | 29 years x 29 countries | filled: 362/841 (43.0%)
  Saved sheet: Silent_Generation         | 29 years x 29 countries | filled: 255/841 (30.3%)
  Saved sheet: Baby_Boomers              | 29 years x 29 countries | filled: 328/841 (39.0%)
  Saved sheet: Generation_X              | 29 years x 29 countries | filled: 327/841 (38.9%)
  Saved sheet: Millennials               | 29 years x 29 countries | filled: 269/841 (32.0%)
  Sa

In [5]:
# ── CHECKPOINT 5: Spot-check pivots ───────────────────────────────────────
print('── CHECKPOINT 5: Pivot spot-checks ──\n')
for group, pivot in pivots.items():
    missing_pct = pivot.isna().sum().sum() / pivot.size * 100
    print(f'{group:25s} | shape: {str(pivot.shape):12s} | missing: {missing_pct:.1f}%')

# Show one pivot as example
if 'Coalition' in pivots:
    print('\nSample — Coalition pivot (first 5 years, first 5 countries):')
    print(pivots['Coalition'].iloc[:5, :5].round(4))


── CHECKPOINT 5: Pivot spot-checks ──

Coalition                 | shape: (29, 29)     | missing: 57.6%
Opposition                | shape: (29, 29)     | missing: 63.5%
Male                      | shape: (29, 29)     | missing: 57.0%
Female                    | shape: (29, 29)     | missing: 57.0%
Silent_Generation         | shape: (29, 29)     | missing: 69.7%
Baby_Boomers              | shape: (29, 29)     | missing: 61.0%
Generation_X              | shape: (29, 29)     | missing: 61.1%
Millennials               | shape: (29, 29)     | missing: 68.0%
Generation_Z              | shape: (29, 29)     | missing: 98.6%
Left                      | shape: (29, 29)     | missing: 57.1%
Centre                    | shape: (29, 29)     | missing: 74.4%
Right                     | shape: (29, 29)     | missing: 56.8%

Sample — Coalition pivot (first 5 years, first 5 countries):
Country      AT      BA  BE  BG  CZ
Year                               
1996     0.3131     NaN NaN NaN NaN
1997     0.

In [6]:
# ── PLOT HELPERS ──────────────────────────────────────────────────────────

COLORS = plt.cm.tab20.colors

def plot_heatmap_sidebyside(dim, groups, pivots, output_dir):
    """
    Side-by-side heatmaps matching the reference style:
      - BuGn colormap, white = missing
      - Countries on y-axis, Years on x-axis
      - Shared colour scale within dimension for fair comparison
      - All years and countries shown even if no data (NaN = white cell)
    """
    available = [g for g in groups if g in pivots]
    n = len(available)
    if n == 0:
        print(f'  No data for {dim}')
        return

    # Union of all years and countries across groups in this dimension
    all_years_dim     = sorted(set(yr for g in available for yr in pivots[g].index))
    all_countries_dim = sorted(set(c  for g in available for c  in pivots[g].columns))

    n_years     = len(all_years_dim)
    n_countries = len(all_countries_dim)

    # Shared colour scale across groups — ignore NaN
    all_vals = np.concatenate([
        pivots[g].reindex(index=all_years_dim, columns=all_countries_dim).values.flatten()
        for g in available
    ])
    vmin = np.nanmin(all_vals)
    vmax = np.nanmax(all_vals)

    # Figure sizing: width per panel scales with years; height scales with countries
    panel_w = max(6, n_years * 0.28)
    panel_h = max(5, n_countries * 0.32)
    fig, axes = plt.subplots(1, n,
                             figsize=(panel_w * n + 1.5, panel_h),
                             dpi=200)
    if n == 1:
        axes = [axes]

    for ax, group in zip(axes, available):
        # Reindex this group's pivot to the dimension-wide union index
        pivot = (
            pivots[group]
            .reindex(index=all_years_dim, columns=all_countries_dim)
        )
        # data: rows = countries (y), cols = years (x)
        data = pivot.T.values

        # Mask NaN so they render white
        masked = np.ma.masked_invalid(data)
        cmap = plt.cm.BuGn.copy()
        cmap.set_bad(color='white')

        im = ax.imshow(
            masked,
            aspect='auto',
            interpolation='nearest',
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            origin='upper'
        )

        # X-axis: years
        ax.set_xticks(range(n_years))
        ax.set_xticklabels(all_years_dim, rotation=45, ha='right', fontsize=7)
        ax.set_xlabel('Year', fontsize=10)

        # Y-axis: countries
        ax.set_yticks(range(n_countries))
        ax.set_yticklabels(all_countries_dim, fontsize=8)
        ax.set_ylabel('Country Code', fontsize=10)

        ax.set_title(group.replace('_', ' '),
                     fontsize=13, fontweight='bold', pad=10)

        # Colourbar on right of each panel
        cbar = plt.colorbar(im, ax=ax, shrink=0.6, pad=0.02)
        cbar.set_label('Mean Value', fontsize=9)
        cbar.ax.tick_params(labelsize=8)

    fig.suptitle(
        f'Heatmap of Mean Values — {dim.capitalize()} (Years x Countries)',
        fontsize=14, fontweight='bold', y=1.01
    )
    plt.tight_layout()
    out = output_dir / f'heatmap_{dim}.png'
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {out.name}')


def plot_linegraph_sidebyside(dim, groups, pivots, output_dir):
    """
    Side-by-side line graphs for all groups in a dimension.
    One line per country, x=Year, y=Mean similarity.
    Missing years create gaps in the line (no interpolation).
    """
    available = [g for g in groups if g in pivots]
    n = len(available)
    if n == 0:
        return

    fig, axes = plt.subplots(1, n, figsize=(9 * n, 6), dpi=200, sharey=True)
    if n == 1:
        axes = [axes]

    for ax, group in zip(axes, available):
        pivot = pivots[group]
        countries = pivot.columns.tolist()
        for i, country in enumerate(countries):
            col = pivot[country]          # keep NaN → produces gaps
            ax.plot(col.index, col.values,
                    marker='o', markersize=3, linewidth=1.2,
                    color=COLORS[i % len(COLORS)],
                    label=country, alpha=0.85)
        ax.set_title(group.replace('_', ' '), fontsize=13, fontweight='bold')
        ax.set_xlabel('Year', fontsize=10)
        ax.set_ylabel('Mean Similarity Score', fontsize=10)
        ax.tick_params(axis='x', rotation=45)
        ax.grid(True, linestyle='--', alpha=0.4)
        ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left',
                  fontsize=7, ncol=1, frameon=True)

    fig.suptitle(
        f'Line Graph — {dim.capitalize()}: Year-wise Average Nature-Importance Similarity',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    out = output_dir / f'linegraph_{dim}.png'
    plt.savefig(out, dpi=200, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {out.name}')

print('Plot helper functions defined ✓')


Plot helper functions defined ✓


In [7]:
# ── STEP 3: Generate 4 heatmaps (one per dimension) ───────────────────────
print('Generating heatmaps…')
for dim, groups in DIMENSIONS.items():
    plot_heatmap_sidebyside(dim, groups, pivots, OUTPUT_DIR)

print('\n── CHECKPOINT 6: Heatmaps saved ──')
saved = list(OUTPUT_DIR.glob('heatmap_*.png'))
print(f'Files saved: {[f.name for f in saved]}')
assert len(saved) == 4, f'Expected 4 heatmaps, got {len(saved)}'
print('All 4 heatmaps present ✓')


Generating heatmaps…
  Saved: heatmap_alignment.png
  Saved: heatmap_gender.png
  Saved: heatmap_generation.png
  Saved: heatmap_ideology.png

── CHECKPOINT 6: Heatmaps saved ──
Files saved: ['heatmap_ideology.png', 'heatmap_gender.png', 'heatmap_alignment.png', 'heatmap_generation.png']
All 4 heatmaps present ✓


In [8]:
# ── STEP 4: Generate 4 line graphs (one per dimension) ────────────────────
print('Generating line graphs…')
for dim, groups in DIMENSIONS.items():
    plot_linegraph_sidebyside(dim, groups, pivots, OUTPUT_DIR)

print('\n── CHECKPOINT 7: Line graphs saved ──')
saved = list(OUTPUT_DIR.glob('linegraph_*.png'))
print(f'Files saved: {[f.name for f in saved]}')
assert len(saved) == 4, f'Expected 4 line graphs, got {len(saved)}'
print('All 4 line graphs present ✓')


Generating line graphs…
  Saved: linegraph_alignment.png
  Saved: linegraph_gender.png
  Saved: linegraph_generation.png
  Saved: linegraph_ideology.png

── CHECKPOINT 7: Line graphs saved ──
Files saved: ['linegraph_alignment.png', 'linegraph_ideology.png', 'linegraph_gender.png', 'linegraph_generation.png']
All 4 line graphs present ✓


In [9]:
# ── FINAL CHECKPOINT: Summary of all outputs ──────────────────────────────
print('=' * 55)
print('FINAL OUTPUT SUMMARY')
print('=' * 55)
all_outputs = list(OUTPUT_DIR.iterdir())
for f in sorted(all_outputs):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:45s} {size_kb:7.1f} KB')

print(f'\nTotal files in output folder: {len(all_outputs)}')
print(f'Expected: 1 Excel + 4 heatmaps + 4 line graphs = 9 files')
assert len(all_outputs) >= 9, 'Some output files are missing — check above!'
print('\nAll outputs verified ✓')
print(f'\nOutput location: {OUTPUT_DIR}')


FINAL OUTPUT SUMMARY
  heatmap_alignment.png                           127.4 KB
  heatmap_gender.png                              122.8 KB
  heatmap_generation.png                          195.4 KB
  heatmap_ideology.png                            145.0 KB
  linegraph_alignment.png                         536.8 KB
  linegraph_gender.png                            547.3 KB
  linegraph_generation.png                       1086.3 KB
  linegraph_ideology.png                          767.0 KB
  meta_average_analysis.ipynb                      52.3 KB
  meta_averages_by_group.xlsx                      77.6 KB

Total files in output folder: 10
Expected: 1 Excel + 4 heatmaps + 4 line graphs = 9 files

All outputs verified ✓

Output location: /Users/harshit/Desktop/KelloggXParlamint/Kellogg/data/results/meta_transcript/Analysis


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# LOW COMPLEXITY ANALYSES  (1–3)
# 1. Divergence from dimension mean
# 2. Gap series (group A minus group B per country per year)
# 3. LOESS trend lines per group
# ════════════════════════════════════════════════════════════════════════════
import statsmodels.api as sm
from matplotlib.lines   import Line2D
from matplotlib.patches import Patch

LOW_OUT = OUTPUT_DIR / 'low_complexity'
LOW_OUT.mkdir(exist_ok=True)

BREAKPOINTS = {
    'Paris Agreement (2015)': 2015,
    'COVID disruption (2020)': 2020,
    'Energy crisis (2022)':   2022,
}
GAP_PAIRS = {
    'alignment':  ('Coalition', 'Opposition'),
    'gender':     ('Male',      'Female'),
    'ideology':   ('Left',      'Right'),
    'generation': ('Millennials','Baby_Boomers'),
}

def loess_smooth(x, y, frac=0.35):
    mask = ~np.isnan(y)
    if mask.sum() < 4:
        return np.full_like(y, np.nan, dtype=float)
    smoothed = sm.nonparametric.lowess(y[mask], x[mask], frac=frac, return_sorted=False)
    result = np.full_like(y, np.nan, dtype=float)
    result[mask] = smoothed
    return result

# ── ANALYSIS 1 & 3 combined: Divergence + LOESS per dimension ───────────────
print('Running Analysis 1 & 3: Divergence from mean + LOESS trends...')
for dim, groups in DIMENSIONS.items():
    available = [g for g in groups if g in pivots]
    if len(available) < 2:
        continue
    n = len(available)
    fig, axes = plt.subplots(2, n, figsize=(7*n, 10), dpi=180)
    if n == 1:
        axes = axes.reshape(2, 1)

    # Row 0: LOESS raw trends
    # Row 1: Divergence from dimension mean
    dim_mean = pd.concat([pivots[g] for g in available]).groupby(level=0).mean()

    for col_i, group in enumerate(available):
        pivot = pivots[group]
        years_arr = np.array(pivot.index, dtype=float)

        # Row 0 — LOESS
        ax0 = axes[0, col_i]
        for i, country in enumerate(pivot.columns):
            raw = pivot[country].values.astype(float)
            ax0.scatter(pivot.index, raw, s=10, alpha=0.35,
                        color=COLORS[i % len(COLORS)])
            sm_vals = loess_smooth(years_arr, raw)
            ax0.plot(pivot.index, sm_vals, linewidth=1.6,
                     color=COLORS[i % len(COLORS)], label=country)
        ax0.set_title(f'{group.replace("_"," ")} — LOESS Trends',
                      fontsize=11, fontweight='bold')
        ax0.set_xlabel('Year', fontsize=9)
        ax0.set_ylabel('Mean Similarity', fontsize=9)
        ax0.tick_params(axis='x', rotation=45)
        ax0.grid(True, linestyle='--', alpha=0.3)
        ax0.legend(bbox_to_anchor=(1.01,1), loc='upper left',
                   fontsize=6, ncol=1, frameon=True)

        # Row 1 — Divergence
        ax1 = axes[1, col_i]
        divergence = pivot.subtract(dim_mean, axis=0)
        for i, country in enumerate(divergence.columns):
            vals = divergence[country].values.astype(float)
            ax1.plot(divergence.index, vals, linewidth=1.2, marker='o',
                     markersize=3, color=COLORS[i % len(COLORS)],
                     label=country, alpha=0.8)
        ax1.axhline(0, color='black', linewidth=1.2, linestyle='--')
        ax1.set_title(f'{group.replace("_"," ")} — Divergence from Dim. Mean',
                      fontsize=11, fontweight='bold')
        ax1.set_xlabel('Year', fontsize=9)
        ax1.set_ylabel('Score − Dimension Mean', fontsize=9)
        ax1.tick_params(axis='x', rotation=45)
        ax1.grid(True, linestyle='--', alpha=0.3)
        ax1.legend(bbox_to_anchor=(1.01,1), loc='upper left',
                   fontsize=6, ncol=1, frameon=True)

    fig.suptitle(f'{dim.capitalize()}: LOESS Trends (top) & Divergence from Mean (bottom)',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    out = LOW_OUT / f'loess_divergence_{dim}.png'
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {out.name}')

# ── ANALYSIS 2: Gap series ───────────────────────────────────────────────────
print('\nRunning Analysis 2: Gap series...')
fig, axes = plt.subplots(2, 2, figsize=(18, 10), dpi=180)
axes = axes.flatten()
for ax_i, (dim, (g1, g2)) in enumerate(GAP_PAIRS.items()):
    ax = axes[ax_i]
    if g1 not in pivots or g2 not in pivots:
        ax.set_visible(False)
        continue
    gap = pivots[g1].subtract(pivots[g2])
    for i, country in enumerate(gap.columns):
        vals = gap[country].values.astype(float)
        ax.plot(gap.index, vals, linewidth=1.2, marker='o', markersize=3,
                color=COLORS[i % len(COLORS)], label=country, alpha=0.8)
    ax.axhline(0, color='black', linewidth=1.5, linestyle='--')
    ax.fill_between(gap.index,
                    gap.mean(axis=1), 0,
                    alpha=0.12, color='steelblue', label='Cross-country mean gap')
    ax.set_title(f'{dim.capitalize()}: {g1.replace("_"," ")} − {g2.replace("_"," ")}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Year', fontsize=9)
    ax.set_ylabel('Score Gap', fontsize=9)
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.legend(bbox_to_anchor=(1.01,1), loc='upper left',
              fontsize=6, ncol=1, frameon=True)
fig.suptitle('Gap Series: Group A − Group B per Dimension (positive = Group A leads)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
out = LOW_OUT / 'gap_series_all_dimensions.png'
plt.savefig(out, dpi=180, bbox_inches='tight')
plt.close()
print(f'  Saved: {out.name}')

print('\n── CHECKPOINT LOW: All low-complexity analyses complete ──')
saved = list(LOW_OUT.glob('*.png'))
print(f'Files saved ({len(saved)}): {[f.name for f in sorted(saved)]}')


Running Analysis 1 & 3: Divergence from mean + LOESS trends...
  Saved: loess_divergence_alignment.png
  Saved: loess_divergence_gender.png
  Saved: loess_divergence_generation.png
  Saved: loess_divergence_ideology.png

Running Analysis 2: Gap series...
  Saved: gap_series_all_dimensions.png

Running Analysis 4: Pre/Post event...


/var/folders/sf/rmvwvxcn68g4drd3v0z0y8vw0000gn/T/ipykernel_53360/2082149127.py:148: RuntimeWarning: Mean of empty slice
  pre_means.append(np.nanmean(pre))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


  Saved: prepost_Paris_Agreement_2015.png
  Saved: prepost_COVID_disruption_2020.png
  Saved: prepost_Energy_crisis_2022.png

── CHECKPOINT LOW: All low-complexity analyses complete ──
Files saved (8): ['gap_series_all_dimensions.png', 'loess_divergence_alignment.png', 'loess_divergence_gender.png', 'loess_divergence_generation.png', 'loess_divergence_ideology.png', 'prepost_COVID_disruption_2020.png', 'prepost_Energy_crisis_2022.png', 'prepost_Paris_Agreement_2015.png']


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MEDIUM ANALYSIS 4: Rolling 3-year correlation between groups
# For each country, how synchronised are the two groups in a dimension?
# High corr = groups move together; low/negative = diverging behaviour
# ════════════════════════════════════════════════════════════════════════════
MED_OUT = OUTPUT_DIR / 'medium_complexity'
MED_OUT.mkdir(exist_ok=True)

ROLL_PAIRS = {
    'alignment':  ('Coalition',  'Opposition'),
    'gender':     ('Male',       'Female'),
    'ideology':   ('Left',       'Right'),
    'generation': ('Millennials','Baby_Boomers'),
}
WINDOW = 3

print('Running Analysis 5: Rolling correlation...')

for dim, (g1, g2) in ROLL_PAIRS.items():
    if g1 not in pivots or g2 not in pivots:
        print(f'  [SKIP] {dim} — missing pivot')
        continue

    p1 = pivots[g1]
    p2 = pivots[g2]
    common_countries = [c for c in p1.columns if c in p2.columns]
    common_years     = sorted(set(p1.index) & set(p2.index))

    # Build rolling correlation matrix: shape (years x countries)
    roll_corr = pd.DataFrame(index=common_years, columns=common_countries, dtype=float)
    for country in common_countries:
        s1 = p1.loc[common_years, country]
        s2 = p2.loc[common_years, country]
        roll_corr[country] = s1.rolling(WINDOW, min_periods=2).corr(s2)

    # ── Plot: heatmap + mean line side by side
    fig = plt.figure(figsize=(18, 7), dpi=180)
    gs  = fig.add_gridspec(1, 2, width_ratios=[2, 1], wspace=0.35)
    ax_heat = fig.add_subplot(gs[0])
    ax_line = fig.add_subplot(gs[1])

    # Heatmap
    data   = roll_corr.T.values.astype(float)
    masked = np.ma.masked_invalid(data)
    cmap_div = plt.cm.RdYlGn.copy()
    cmap_div.set_bad('lightgrey')
    im = ax_heat.imshow(masked, aspect='auto', interpolation='nearest',
                        cmap=cmap_div, vmin=-1, vmax=1, origin='upper')
    ax_heat.set_xticks(range(len(common_years)))
    ax_heat.set_xticklabels(common_years, rotation=45, ha='right', fontsize=7)
    ax_heat.set_yticks(range(len(common_countries)))
    ax_heat.set_yticklabels(common_countries, fontsize=8)
    ax_heat.set_xlabel('Year', fontsize=10)
    ax_heat.set_ylabel('Country', fontsize=10)
    ax_heat.set_title(f'Rolling {WINDOW}-yr Correlation\n{g1.replace("_"," ")} vs {g2.replace("_"," ")}',
                      fontsize=11, fontweight='bold')
    cbar = plt.colorbar(im, ax=ax_heat, shrink=0.7)
    cbar.set_label('Pearson r', fontsize=9)

    # Mean line across countries
    mean_corr = roll_corr.mean(axis=1)
    ax_line.plot(mean_corr.index, mean_corr.values,
                 color='steelblue', linewidth=2, marker='o', markersize=4)
    ax_line.fill_between(mean_corr.index,
                         roll_corr.min(axis=1), roll_corr.max(axis=1),
                         alpha=0.15, color='steelblue', label='Country range')
    ax_line.axhline(0, color='black', linewidth=1, linestyle='--')
    ax_line.axhline(0.5,  color='green', linewidth=0.8, linestyle=':', alpha=0.6)
    ax_line.axhline(-0.5, color='red',   linewidth=0.8, linestyle=':', alpha=0.6)
    ax_line.set_xlabel('Year', fontsize=10)
    ax_line.set_ylabel('Mean Rolling Correlation', fontsize=10)
    ax_line.set_title('Cross-country Mean\n(shading = country range)', fontsize=11, fontweight='bold')
    ax_line.tick_params(axis='x', rotation=45)
    ax_line.grid(True, linestyle='--', alpha=0.3)
    ax_line.set_ylim(-1.05, 1.05)
    ax_line.legend(fontsize=8)

    fig.suptitle(f'Medium Analysis 5 — Rolling Correlation: {dim.capitalize()}',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    out = MED_OUT / f'rolling_corr_{dim}.png'
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {out.name}')

print('\n── CHECKPOINT M1: Rolling correlation complete ──')
print(f'Files: {[f.name for f in sorted(MED_OUT.glob("rolling_corr_*.png"))]}')


Running Analysis 5: Rolling correlation...


/var/folders/sf/rmvwvxcn68g4drd3v0z0y8vw0000gn/T/ipykernel_53360/1636047866.py:80: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: rolling_corr_alignment.png


/var/folders/sf/rmvwvxcn68g4drd3v0z0y8vw0000gn/T/ipykernel_53360/1636047866.py:80: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: rolling_corr_gender.png


/var/folders/sf/rmvwvxcn68g4drd3v0z0y8vw0000gn/T/ipykernel_53360/1636047866.py:80: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: rolling_corr_ideology.png


/var/folders/sf/rmvwvxcn68g4drd3v0z0y8vw0000gn/T/ipykernel_53360/1636047866.py:80: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


  Saved: rolling_corr_generation.png

── CHECKPOINT M1: Rolling correlation complete ──
Files: ['rolling_corr_alignment.png', 'rolling_corr_gender.png', 'rolling_corr_generation.png', 'rolling_corr_ideology.png']


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MEDIUM ANALYSIS 5: Variance decomposition
# For each year: between-country variance vs between-group variance
# Rising between-group = groups polarising
# Rising between-country = geographic spread dominating
# ════════════════════════════════════════════════════════════════════════════
print('Running Analysis 6: Variance decomposition...')

for dim, groups in DIMENSIONS.items():
    available = [g for g in groups if g in pivots]
    if len(available) < 2:
        continue

    all_years_dim = sorted(set(yr for g in available for yr in pivots[g].index))

    var_between_countries = []   # variance across countries, averaged over groups
    var_between_groups    = []   # variance across groups, averaged over countries
    years_used            = []

    for yr in all_years_dim:
        # Collect scores: shape (n_groups x n_countries)
        rows = []
        for g in available:
            if yr in pivots[g].index:
                rows.append(pivots[g].loc[yr])
        if len(rows) < 2:
            continue
        mat = pd.DataFrame(rows, index=[g for g in available if yr in pivots[g].index])
        # between countries: variance across columns for each group, then mean
        bc = mat.var(axis=1, ddof=1).mean()   # mean of per-group country variance
        # between groups: variance across rows for each country, then mean
        bg = mat.var(axis=0, ddof=1).mean()   # mean of per-country group variance
        var_between_countries.append(bc)
        var_between_groups.append(bg)
        years_used.append(yr)

    var_bc = np.array(var_between_countries)
    var_bg = np.array(var_between_groups)

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=180)

    # Left: raw variance lines
    ax = axes[0]
    ax.plot(years_used, var_bc, color='steelblue', linewidth=2,
            marker='o', markersize=4, label='Between-country variance')
    ax.plot(years_used, var_bg, color='coral', linewidth=2,
            marker='s', markersize=4, label='Between-group variance')
    ax.set_xlabel('Year', fontsize=10)
    ax.set_ylabel('Variance', fontsize=10)
    ax.set_title('Raw Variance Over Time', fontsize=11, fontweight='bold')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.legend(fontsize=9)

    # Right: stacked area — proportion of each variance type
    ax2 = axes[1]
    total = var_bc + var_bg
    total = np.where(total == 0, np.nan, total)
    pct_bc = var_bc / total * 100
    pct_bg = var_bg / total * 100
    ax2.stackplot(years_used, pct_bc, pct_bg,
                  labels=['Between-country %', 'Between-group %'],
                  colors=['steelblue', 'coral'], alpha=0.75)
    ax2.axhline(50, color='black', linewidth=1, linestyle='--', alpha=0.5)
    ax2.set_xlabel('Year', fontsize=10)
    ax2.set_ylabel('% of Total Variance', fontsize=10)
    ax2.set_title('Variance Share (stacked)', fontsize=11, fontweight='bold')
    ax2.set_ylim(0, 100)
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, linestyle='--', alpha=0.3)
    ax2.legend(fontsize=9, loc='upper left')

    fig.suptitle(f'Medium Analysis 6 — Variance Decomposition: {dim.capitalize()}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    out = MED_OUT / f'variance_decomp_{dim}.png'
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {out.name}')

print('\n── CHECKPOINT M2: Variance decomposition complete ──')
print(f'Files: {[f.name for f in sorted(MED_OUT.glob("variance_decomp_*.png"))]}')


Running Analysis 6: Variance decomposition...
  Saved: variance_decomp_alignment.png
  Saved: variance_decomp_gender.png
  Saved: variance_decomp_generation.png
  Saved: variance_decomp_ideology.png

── CHECKPOINT M2: Variance decomposition complete ──
Files: ['variance_decomp_alignment.png', 'variance_decomp_gender.png', 'variance_decomp_generation.png', 'variance_decomp_ideology.png']


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# MEDIUM ANALYSIS 6: Rank stability — bump charts
# For each year, rank groups within a dimension by mean score across countries
# Tracks whether rank order is stable or groups swap positions over time
# ════════════════════════════════════════════════════════════════════════════
print('Running Analysis 7: Rank stability bump charts...')

for dim, groups in DIMENSIONS.items():
    available = [g for g in groups if g in pivots]
    if len(available) < 2:
        continue

    all_years_dim = sorted(set(yr for g in available for yr in pivots[g].index))

    # Mean score per group per year (averaged across countries)
    mean_scores = pd.DataFrame(
        {g: pivots[g].mean(axis=1) for g in available}
    ).reindex(all_years_dim)

    # Rank: 1 = highest score in that year
    ranked = mean_scores.rank(axis=1, ascending=False, method='min')

    n_groups = len(available)
    group_colors = {g: COLORS[i % len(COLORS)] for i, g in enumerate(available)}

    fig, axes = plt.subplots(1, 2, figsize=(18, 6), dpi=180,
                             gridspec_kw={'width_ratios': [2, 1]})

    # Left: Bump chart
    ax_bump = axes[0]
    for group in available:
        r = ranked[group].dropna()
        ax_bump.plot(r.index, r.values,
                     linewidth=2.5, marker='o', markersize=7,
                     color=group_colors[group],
                     label=group.replace('_',' '))
        # Label at end
        if len(r) > 0:
            ax_bump.annotate(group.replace('_',' '),
                             xy=(r.index[-1], r.values[-1]),
                             xytext=(3, 0), textcoords='offset points',
                             fontsize=8, color=group_colors[group], va='center')
    ax_bump.invert_yaxis()   # rank 1 at top
    ax_bump.set_yticks(range(1, n_groups+1))
    ax_bump.set_yticklabels([f'Rank {i}' for i in range(1, n_groups+1)], fontsize=9)
    ax_bump.set_xlabel('Year', fontsize=10)
    ax_bump.set_title('Bump Chart — Rank by Year\n(Rank 1 = highest framing score)',
                      fontsize=11, fontweight='bold')
    ax_bump.tick_params(axis='x', rotation=45)
    ax_bump.grid(True, axis='y', linestyle='--', alpha=0.3)
    ax_bump.legend(bbox_to_anchor=(1.01,1), loc='upper left', fontsize=8)

    # Right: Rank-change heatmap (how much does each group's rank change YoY)
    ax_heat = axes[1]
    rank_change = ranked.diff().fillna(0)
    data   = rank_change[available].T.values.astype(float)
    masked = np.ma.masked_invalid(data)
    cmap_rc = plt.cm.RdYlGn_r.copy()
    cmap_rc.set_bad('lightgrey')
    im = ax_heat.imshow(masked, aspect='auto', interpolation='nearest',
                        cmap=cmap_rc, vmin=-n_groups+1, vmax=n_groups-1,
                        origin='upper')
    ax_heat.set_xticks(range(len(all_years_dim)))
    ax_heat.set_xticklabels(all_years_dim, rotation=45, ha='right', fontsize=7)
    ax_heat.set_yticks(range(len(available)))
    ax_heat.set_yticklabels([g.replace('_',' ') for g in available], fontsize=9)
    ax_heat.set_title('Year-on-Year Rank Change\n(red = dropped, green = rose)',
                      fontsize=11, fontweight='bold')
    cbar = plt.colorbar(im, ax=ax_heat, shrink=0.7)
    cbar.set_label('Rank change', fontsize=9)

    fig.suptitle(f'Medium Analysis 7 — Rank Stability: {dim.capitalize()}',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    out = MED_OUT / f'rank_stability_{dim}.png'
    plt.savefig(out, dpi=180, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {out.name}')

print('\n── CHECKPOINT M3: Rank stability complete ──')
print(f'Files: {[f.name for f in sorted(MED_OUT.glob("rank_stability_*.png"))]}')


Running Analysis 7: Rank stability bump charts...
  Saved: rank_stability_alignment.png
  Saved: rank_stability_gender.png
  Saved: rank_stability_generation.png
  Saved: rank_stability_ideology.png

── CHECKPOINT M3: Rank stability complete ──
Files: ['rank_stability_alignment.png', 'rank_stability_gender.png', 'rank_stability_generation.png', 'rank_stability_ideology.png']


In [14]:
# ════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY — all outputs across all analyses
# ════════════════════════════════════════════════════════════════════════════
print('=' * 65)
print('COMPLETE OUTPUT SUMMARY')
print('=' * 65)

sections = {
    'Core outputs (heatmaps + line graphs)': OUTPUT_DIR,
    'Low complexity analyses':              LOW_OUT,
    'Medium complexity analyses':           MED_OUT,
}

total_files = 0
for section, folder in sections.items():
    files = sorted([f for f in folder.iterdir() if f.is_file()])
    print(f'\n  {section} ({len(files)} files)')
    print(f'  {"─" * 55}')
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f'    {f.name:50s} {size_kb:7.1f} KB')
    total_files += len(files)

print(f'\n{"=" * 65}')
print(f'Total files generated: {total_files}')
print(f'Output root: {OUTPUT_DIR}')
print('=' * 65)


COMPLETE OUTPUT SUMMARY

  Core outputs (heatmaps + line graphs) (10 files)
  ───────────────────────────────────────────────────────
    heatmap_alignment.png                                127.4 KB
    heatmap_gender.png                                   122.8 KB
    heatmap_generation.png                               195.4 KB
    heatmap_ideology.png                                 145.0 KB
    linegraph_alignment.png                              536.8 KB
    linegraph_gender.png                                 547.3 KB
    linegraph_generation.png                            1086.3 KB
    linegraph_ideology.png                               767.0 KB
    meta_average_analysis.ipynb                           54.3 KB
    meta_averages_by_group.xlsx                           77.6 KB

  Low complexity analyses (8 files)
  ───────────────────────────────────────────────────────
    gap_series_all_dimensions.png                       1141.5 KB
    loess_divergence_alignment.png           